# Weird AI Attention Mechanism Exploration

In this notebook, you will manually compute attention scores, attention weights, and context vectors before implementing the same ideas in `attention.py`.

The goal is to understand the math behind attention before hiding it inside reusable PyTorch classes.

## Section 1: Create Toy Embeddings

Use a small manually created tensor.
 - Each row represents one token embedding. Each column represents one feature of that token.
 - For this notebook, these are hard-coded embeddings. Later, Weird AI will learn embeddings from lyric tokens.

Example:
```python
import torch

torch.manual_seed(123)

inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],
        [0.55, 0.87, 0.66],
        [0.57, 0.85, 0.64],
        [0.22, 0.58, 0.33],
    ]
)

print(inputs)
print(inputs.shape)
```

In [ ]:
import torch

torch.manual_seed(123)

inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],
        [0.55, 0.87, 0.66],
        [0.57, 0.85, 0.64],
        [0.22, 0.58, 0.33],
    ]
)

print(inputs)
print(inputs.shape)

## Section 2: Choose One Query Token
Start by calculating attention for one token.

The selected token acts as the query.  We are asking: "How much should this token pay attention to each token in the sequence?"

In [ ]:
query_index = 1
query = inputs[query_index]

print(query)
print(query.shape)

## Section 3: Compute Attention Scores

Compute dot products between one query token and all input tokens.
 - Conceptual question: Why does a dot product help estimate similarity between two token vectors?

 Note: The attention scores are raw similarity values. Larger dot products suggest stronger similarity between the query token and another token.

In [ ]:
attention_scores = torch.empty(inputs.shape[0])

for index, token_embedding in enumerate(inputs):
    attention_scores[index] = torch.dot(query, token_embedding)

print(attention_scores)

In [ ]:
# Here is a vectorized version
attention_scores_vectorized = inputs @ query

print(attention_scores_vectorized)

## Section 4: Normalize with Softmax

Convert raw attention scores into attention weights.
 - Be sure to verify that the sum of the weights is approximately 1.0

In [ ]:
attention_weights = torch.softmax(attention_scores, dim=0)

print(attention_weights)
print(attention_weights.sum())

## Section 5: Compute a Context Vector

The context vector is a weighted mixture of the input token embeddings.

Tokens with larger attention weights contribute more to the final context vector.

In [ ]:
context_vector = torch.zeros(query.shape)

for index, token_embedding in enumerate(inputs):
    context_vector += attention_weights[index] * token_embedding

print(context_vector)

In [ ]:
# Vectorize the context_vector
context_vector_vectorized = attention_weights @ inputs

print(context_vector_vectorized)

## Section 6: Compute Self-Attention for All Tokens

Instead of calculating attention for only one token, we can calculate attention for every token at once.

This is the basic idea behind self-attention.

In [ ]:
all_attention_scores = inputs @ inputs.T

print(all_attention_scores)
print(all_attention_scores.shape)

In [ ]:
all_attention_weights = torch.softmax(all_attention_scores, dim=-1)

print(all_attention_weights)
print(all_attention_weights.sum(dim=-1))

In [ ]:
all_context_vectors = all_attention_weights @ inputs

print(all_context_vectors)
print(all_context_vectors.shape)

## Connecting This Notebook to attention.py

The notebook manually calculated attention step by step.

The `SimpleSelfAttention` class in `attention.py` should perform the same operation in reusable form.

In [ ]:
from weird_ai.attention import SimpleSelfAttention

attention = SimpleSelfAttention()

context_vectors, weights = attention(inputs)

print(context_vectors)
print(weights)
print(context_vectors.shape)
print(weights.shape)

## Reflection Questions

1. What does a larger attention score mean?
2. Why do attention weights need to sum to 1?
3. Why is the context vector a weighted mixture of token embeddings?
4. How does self-attention differ from calculating attention for only one query token?
5. How does this connect to lyric generation in Weird AI?